In [6]:
import os
import gc
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights
from torchvision import transforms
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from scipy import stats
from torch.utils.data import TensorDataset, DataLoader

# --- CONFIGURATION VIDÉO ---
CONFIG = {
    'layer_name': 'layer3',   # Couche recommandée pour garder de la nuance sémantique
    'n_random_sets': 10,
    'alpha': 0.01,
    'batch_size': 4,          # TRÈS IMPORTANT: Réduit à 4 car la vidéo consomme 4x à 16x plus de VRAM
    'num_frames': 16,         # Nombre d'images extraites par vidéo (Standard pour R3D_18)
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# --- 1. PRÉPARATION DU MODÈLE VIDÉO ET HOOKS ---
class VideoModelWrapper:
    def __init__(self, model, layer_name):
        self.model = model.to(CONFIG['device'])
        self.model.eval()
        self.layer_name = layer_name
        self.activations = {'value': None}
        self.gradients = {'value': None}
        self.hooks = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations['value'] = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients['value'] = grad_output[0]

        found = False
        for name, module in self.model.named_modules():
            if name == self.layer_name:
                self.hooks.append(module.register_forward_hook(forward_hook))
                self.hooks.append(module.register_full_backward_hook(backward_hook))
                print(f"Hook attaché à la couche 3D: {name}")
                found = True
                break
        if not found:
            raise ValueError(f"Couche {self.layer_name} introuvable.")

    def get_activation(self, inputs):
        dataset = TensorDataset(inputs)
        loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=False)
        all_acts = []
        
        with torch.no_grad():
            for batch in loader:
                # Shape attendue : (B, C, T, H, W)
                batch_vids = batch[0].to(CONFIG['device'])
                _ = self.model(batch_vids)
                act = self.activations['value']
                
                # --- POOLING SPATIO-TEMPOREL ---
                # act.shape est (Batch, Channels, Time, Height, Width)
                if len(act.shape) == 5:
                    # On moyenne sur le temps, la hauteur et la largeur
                    act = torch.mean(act, dim=[2, 3, 4])
                
                all_acts.append(act.detach().cpu().numpy())
                
                self.activations['value'] = None 
                del batch_vids, act
        
        torch.cuda.empty_cache()
        return np.concatenate(all_acts)

    def get_gradient(self, vid_tensor, target_class_index):
        vid_tensor = vid_tensor.to(CONFIG['device'])
        self.model.zero_grad()
        outputs = self.model(vid_tensor)
        
        one_hot = torch.zeros_like(outputs)
        one_hot[0, target_class_index] = 1
        outputs.backward(gradient=one_hot, retain_graph=True)
        
        grad = self.gradients['value']
        
        # --- POOLING SPATIO-TEMPOREL DU GRADIENT ---
        if len(grad.shape) == 5:
            grad = torch.mean(grad, dim=[2, 3, 4])
            
        grad_cpu = grad.detach().cpu().numpy()
        
        self.gradients['value'] = None
        del vid_tensor, outputs, one_hot, grad
        return grad_cpu

    def clear_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []


# --- 2. EXTRACTION ET PRÉTRAITEMENT VIDÉO ---
def load_videos_from_folder(folder, max_vids=50):
    """Charge les vidéos, extrait 16 frames uniformément, et renvoie un Tenseur (N, C, T, H, W)."""
    videos = []
    if not os.path.exists(folder):
        print(f"Attention: Dossier introuvable {folder}")
        return None
        
    for filename in os.listdir(folder)[:max_vids]:
        vid_path = os.path.join(folder, filename)
        if not vid_path.endswith(('.mp4', '.avi', '.mov')):
            continue
            
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames < CONFIG['num_frames']:
            cap.release()
            continue # Ignorer les vidéos trop courtes
            
        # Échantillonnage uniforme des frames
        indices = np.linspace(0, total_frames - 1, CONFIG['num_frames'], dtype=int)
        frames = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (112, 112)) # Standard pour R3D_18
                frames.append(frame)
        cap.release()
        
        if len(frames) == CONFIG['num_frames']:
            # Numpy shape: (T, H, W, C)
            vid_np = np.array(frames, dtype=np.float32) / 255.0
            # Convertir en Tenseur (C, T, H, W) attendu par PyTorch 3D
            vid_tensor = torch.tensor(vid_np).permute(3, 0, 1, 2)
            
            # Normalisation Kinetics-400
            mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
            std = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
            vid_tensor = (vid_tensor - mean) / std
            
            videos.append(vid_tensor)
            
    if not videos:
        raise ValueError(f"Dossier vide ou vidéos invalides: {folder}")
        
    return torch.stack(videos)


# --- 3. ENTRAÎNEMENT DU CAV (Identique à l'Image) ---
class CAV:
    def __init__(self, concept_acts, random_acts):
        self.concept_acts = concept_acts
        self.random_acts = random_acts
        self.coef = None
        self.accuracy = 0
        self._train()

    def _train(self):
        min_len = min(len(self.concept_acts), len(self.random_acts))
        c_acts = self.concept_acts[:min_len]
        r_acts = self.random_acts[:min_len]
        
        X = np.concatenate([c_acts, r_acts])
        y = np.concatenate([np.ones(len(c_acts)), np.zeros(len(r_acts))])
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
        lm = SGDClassifier(alpha=CONFIG['alpha'], max_iter=1000, tol=1e-3, loss='hinge')
        lm.fit(X_train, y_train)
        
        self.coef = lm.coef_[0]
        self.accuracy = lm.score(X_test, y_test)


# --- 4. CALCUL DU SCORE TCAV ---
def compute_tcav_score(model_wrapper, target_vids, cav_vec, target_class_idx):
    sensitivities = []
    for i in range(len(target_vids)):
        vid_tensor = target_vids[i].unsqueeze(0)
        grad = model_wrapper.get_gradient(vid_tensor, target_class_idx)
        
        cav_norm = cav_vec / np.linalg.norm(cav_vec)
        sens = np.dot(grad.flatten(), cav_norm.flatten())
        sensitivities.append(sens)
    
    sensitivities = np.array(sensitivities)
    pos_magnitude = np.sum(np.abs(sensitivities[sensitivities > 0]))
    total_magnitude = np.sum(np.abs(sensitivities))
    score_magnitude = pos_magnitude / total_magnitude if total_magnitude > 0 else 0.0
    
    return score_magnitude


# --- 5. PIPELINE PRINCIPAL ---
def run_video_tcav(concept_path, target_path, random_root_path, target_class_idx):
    torch.cuda.empty_cache()
    gc.collect()
    
    print("--- Chargement du modèle ResNet3D-18 (Kinetics-400) ---")
    weights = R3D_18_Weights.KINETICS400_V1
    model = r3d_18(weights=weights)
    wrapper = VideoModelWrapper(model, CONFIG['layer_name'])
    
    try:
        print("--- 1. Extraction des activations du Concept (Vidéos) ---")
        concept_vids = load_videos_from_folder(concept_path)
        concept_acts = wrapper.get_activation(concept_vids)
        del concept_vids; gc.collect()
        
        print("--- 2. Extraction des activations Cible (Vidéos) ---")
        target_vids = load_videos_from_folder(target_path)
        
        tcav_scores_mag = []
        
        print(f"--- 3. Boucle sur {CONFIG['n_random_sets']} ensembles aléatoires ---")
        for i in range(CONFIG['n_random_sets']):
            random_folder = os.path.join(random_root_path, f"random_{i+1}")
            try:
                random_vids = load_videos_from_folder(random_folder)
            except ValueError:
                continue

            random_acts = wrapper.get_activation(random_vids)
            del random_vids
            
            cav = CAV(concept_acts, random_acts)
            if cav.accuracy < 0.65:
                print(f"Warning: CAV faible pour Random_{i} (Acc: {cav.accuracy:.2f})")
                
            score_mag = compute_tcav_score(wrapper, target_vids, cav.coef, target_class_idx)
            tcav_scores_mag.append(score_mag)
            print(f"Run {i}: Score Mag = {score_mag:.4f}")
            
            del random_acts, cav; gc.collect()

        if len(tcav_scores_mag) > 1:
            avg_score = np.mean(tcav_scores_mag)
            std_score = np.std(tcav_scores_mag)
            _, p_val = stats.ttest_1samp(tcav_scores_mag, 0.5)
            
            print("\n--- RÉSULTATS FINAUX VIDÉO ---")
            print(f"Score TCAV Moyen : {avg_score:.4f} ± {std_score:.4f}")
            print(f"P-value : {p_val:.5e}")
            if p_val < 0.05:
                print("RÉSULTAT: Le concept dynamique est SIGNIFICATIF.")
            else:
                print("RÉSULTAT: Non significatif (Bruit contextuel).")
    finally:
        wrapper.clear_hooks()
        del wrapper, model
        torch.cuda.empty_cache()
        gc.collect()



In [7]:
if __name__ == "__main__":
    # Index 195 correspond au 'Long Jump' dans le modèle pré-entraîné R3D_18 (Kinetics-400)
    run_video_tcav(
        concept_path='/kaggle/input/videos-tcav/videos_tcav/concepts/running', 
        target_path='/kaggle/input/videos-tcav/videos_tcav/target/long_jump', 
        random_root_path='/kaggle/input/videos-tcav/videos_tcav/random', 
        target_class_idx=195 
    )

--- Chargement du modèle ResNet3D-18 (Kinetics-400) ---
Hook attaché à la couche 3D: layer3
--- 1. Extraction des activations du Concept (Vidéos) ---
--- 2. Extraction des activations Cible (Vidéos) ---
--- 3. Boucle sur 10 ensembles aléatoires ---
Run 0: Score Mag = 0.0594
Run 1: Score Mag = 0.1837
Run 2: Score Mag = 0.0621
Run 3: Score Mag = 0.2886
Run 4: Score Mag = 0.1938
Run 5: Score Mag = 0.3381
Run 6: Score Mag = 0.1746
Run 7: Score Mag = 0.0000
Run 8: Score Mag = 0.7550
Run 9: Score Mag = 0.3638

--- RÉSULTATS FINAUX VIDÉO ---
Score TCAV Moyen : 0.2419 ± 0.2059
P-value : 4.48423e-03
RÉSULTAT: Le concept dynamique est SIGNIFICATIF.


In [10]:
# --- CONFIGURATION VIDÉO ---
CONFIG = {
    'layer_name': 'layer4',   # Couche recommandée pour garder de la nuance sémantique
    'n_random_sets': 10,
    'alpha': 0.01,
    'batch_size': 4,          # TRÈS IMPORTANT: Réduit à 4 car la vidéo consomme 4x à 16x plus de VRAM
    'num_frames': 16,         # Nombre d'images extraites par vidéo (Standard pour R3D_18)
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

In [11]:
if __name__ == "__main__":
    # Index 195 correspond au 'Long Jump' dans le modèle pré-entraîné R3D_18 (Kinetics-400)
    run_video_tcav(
        concept_path='/kaggle/input/videos-tcav/videos_tcav/concepts/running', 
        target_path='/kaggle/input/videos-tcav/videos_tcav/target/long_jump', 
        random_root_path='/kaggle/input/videos-tcav/videos_tcav/random', 
        target_class_idx=195 
    )

--- Chargement du modèle ResNet3D-18 (Kinetics-400) ---
Hook attaché à la couche 3D: layer4
--- 1. Extraction des activations du Concept (Vidéos) ---
--- 2. Extraction des activations Cible (Vidéos) ---
--- 3. Boucle sur 10 ensembles aléatoires ---
Run 0: Score Mag = 0.0000
Run 1: Score Mag = 0.0000
Run 2: Score Mag = 0.0000
Run 3: Score Mag = 0.0000
Run 4: Score Mag = 0.0000
Run 5: Score Mag = 0.0000
Run 6: Score Mag = 0.0000
Run 7: Score Mag = 0.0000
Run 8: Score Mag = 0.0000
Run 9: Score Mag = 0.0000

--- RÉSULTATS FINAUX VIDÉO ---
Score TCAV Moyen : 0.0000 ± 0.0000
P-value : 0.00000e+00
RÉSULTAT: Le concept dynamique est SIGNIFICATIF.
